# Notebook 02 — Ablation Analysis

Loads all 24 ablation study results and generates:
- Master summary table (all 24 ablations × mean ± std)
- Per-ablation comparison bar charts
- Cross-model oversmoothing comparison (A2 vs D3 vs E3)
- Key insight extraction per ablation group
- SGC accuracy vs K with cosine similarity dual-axis plot (C1)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display

from reddit_gnn.config import RESULTS_ROOT, SEEDS

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

SAVE_DIR = os.path.join(RESULTS_ROOT, 'figures', '02_ablations')
os.makedirs(SAVE_DIR, exist_ok=True)

ALL_ABLATIONS = {
    'graphsage':   {'A1':'Aggregator','A2':'Depth','A3':'Sample Size','A4':'Structure vs Features','A5':'Skip Connections','A6':'Norm Type'},
    'graphsaint':  {'B1':'Sampler Type','B2':'Norm Correction','B3':'Budget','B4':'Walk Params'},
    'sgc':         {'C1':'Hops K','C2':'Norm Scheme','C3':'SGC vs MLP vs GCN'},
    'gat':         {'D1':'Attn Heads','D2':'Attn Dropout','D3':'Depth','D4':'Static Attn Analysis'},
    'gatv2':       {'E1':'Dynamic vs Static','E2':'Shared Weights','E3':'Depth Comparison'},
    'cluster_gcn': {'F1':'Cluster Count','F2':'Diagonal Enhancement','F3':'Clusters/Batch','F4':'METIS vs Random'},
}
print('Ablation groups:', list(ALL_ABLATIONS.keys()))

## 1. Discovery: Load All Ablation Results

In [ ]:
def discover_results(model, ablation_id):
    base = os.path.join(RESULTS_ROOT, model, ablation_id)
    if not os.path.exists(base): return {}
    results = {}
    for variant in sorted(os.listdir(base)):
        seed_metrics = []
        for seed in SEEDS:
            p = os.path.join(base, variant, f'seed{seed}', 'metrics.json')
            if os.path.exists(p):
                with open(p) as f: seed_metrics.append(json.load(f))
        if seed_metrics:
            accs = [m['test_acc'] for m in seed_metrics]
            f1s  = [m.get('f1_macro',0) for m in seed_metrics]
            results[variant] = {'acc_mean': np.mean(accs), 'acc_std': np.std(accs),
                                'f1_mean': np.mean(f1s), 'n': len(accs)}
    return results

all_ablation_data = {}
for model, ablations in ALL_ABLATIONS.items():
    for abl_id in ablations:
        r = discover_results(model, abl_id)
        all_ablation_data[abl_id] = {'model': model, 'results': r, 'name': ALL_ABLATIONS[model][abl_id]}

found = [k for k,v in all_ablation_data.items() if v['results']]
print(f'Found results for {len(found)}/24 ablations: {found}')

## 2. Master Ablation Summary Table

In [ ]:
rows = []
for abl_id, data in sorted(all_ablation_data.items()):
    if not data['results']:
        rows.append({'ID': abl_id, 'Model': data['model'].title(), 'Study': data['name'],
                     'Variants': 0, 'Best Acc': 'N/A', 'Best Variant': 'N/A'})
        continue
    best_v = max(data['results'], key=lambda v: data['results'][v]['acc_mean'])
    best_d = data['results'][best_v]
    rows.append({
        'ID': abl_id, 'Model': data['model'].replace('_',' ').title(),
        'Study': data['name'], 'Variants': len(data['results']),
        'Best Acc': f"{best_d['acc_mean']:.4f} ± {best_d['acc_std']:.4f}",
        'Best Variant': best_v
    })

df_master = pd.DataFrame(rows)
display(df_master.style.set_caption('Master Ablation Summary — 24 Studies').hide(axis='index'))

## 3. Per-Ablation Bar Charts (all available)

In [ ]:
def plot_ablation(abl_id, data):
    results = data['results']
    if not results: return
    variants = list(results.keys())
    means = [results[v]['acc_mean'] for v in variants]
    stds  = [results[v]['acc_std']  for v in variants]
    
    fig, ax = plt.subplots(figsize=(max(8, len(variants)*1.6), 5))
    colors = sns.color_palette('husl', len(variants))
    bars = ax.bar(variants, means, yerr=stds, capsize=4, color=colors, edgecolor='black', lw=0.5)
    ymin = max(0, min(means)-0.04)
    ax.set_ylim(ymin, max(means)+0.02)
    ax.set_ylabel('Test Accuracy', fontsize=11)
    ax.set_title(f'{abl_id}: {data["name"]} ({data["model"]})', fontsize=12)
    plt.xticks(rotation=25, ha='right', fontsize=9)
    for bar, m in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{m:.4f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'{abl_id}.png'), dpi=150)
    plt.show()

for abl_id, data in sorted(all_ablation_data.items()):
    if data['results']:
        plot_ablation(abl_id, data)

## 4. Oversmoothing Comparison: A2 vs D3 vs E3

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
depth_map = {'A2': ('graphsage','GraphSAGE'), 'D3': ('gat','GAT'), 'E3': ('gatv2','GATv2')}

for abl_id, (model, label) in depth_map.items():
    r = discover_results(model, abl_id)
    if not r: continue
    pts = []
    for vname, vd in sorted(r.items()):
        # Extract depth digit from variant name
        parts = vname.rstrip('L').split('-')
        try: depth = int(''.join(c for c in parts[-1] if c.isdigit()))
        except: continue
        pts.append((depth, vd['acc_mean'], vd['acc_std']))
    if pts:
        pts.sort()
        d_arr = [p[0] for p in pts]
        m_arr = [p[1] for p in pts]
        s_arr = [p[2] for p in pts]
        ax.errorbar(d_arr, m_arr, yerr=s_arr, marker='o', label=label, capsize=4, lw=2)

ax.set_xlabel('Number of Layers', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Oversmoothing: Accuracy vs Depth — GraphSAGE, GAT, GATv2', fontsize=13)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'oversmoothing_comparison.png'), dpi=150)
plt.show()

## 5. C1: SGC Accuracy vs K (Oversmoothing Proof)

In [ ]:
import csv as csv_mod

c1_results = discover_results('sgc', 'C1')

# Load oversmoothing log (cosine similarity vs K)
from reddit_gnn.config import SGC_DIR
os_log_path = os.path.join(SGC_DIR, 'sgc_oversmoothing_log.csv')
cos_sim_data = {}
if os.path.exists(os_log_path):
    with open(os_log_path) as f:
        for row in csv_mod.DictReader(f):
            cos_sim_data[int(row['K'])] = float(row['cos_sim'])

if c1_results:
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax2 = ax1.twinx()
    
    ks, accs, stds = [], [], []
    for vname in sorted(c1_results):
        try: k = int(vname.split('-K')[-1])
        except: continue
        ks.append(k)
        accs.append(c1_results[vname]['acc_mean'])
        stds.append(c1_results[vname]['acc_std'])
    
    ax1.errorbar(ks, accs, yerr=stds, color='steelblue', marker='o',
                 label='Test Accuracy', capsize=4, lw=2)
    ax1.set_xlabel('Propagation Hops K', fontsize=12)
    ax1.set_ylabel('Test Accuracy', color='steelblue', fontsize=12)
    
    if cos_sim_data:
        cos_ks = sorted(cos_sim_data)
        cos_vals = [cos_sim_data[k] for k in cos_ks]
        ax2.plot(cos_ks, cos_vals, color='darkorange', marker='s',
                 label='Pairwise Cos Sim', ls='--', lw=2)
        ax2.set_ylabel('Pairwise Cosine Similarity', color='darkorange', fontsize=12)
    
    ax1.set_title('SGC: Accuracy vs Propagation Hops K (C1)', fontsize=13)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR,'C1_oversmoothing.png'), dpi=150)
    plt.show()
else:
    print('C1 results not available yet.')

## 6. F1: ClusterGCN — Accuracy vs Edge Retention

In [ ]:
# Expected edge retention rates from the implementation plan
retention_map = {500: 0.85, 1000: 0.80, 1500: 0.75, 3000: 0.65, 6000: 0.50}
f1_results = discover_results('cluster_gcn', 'F1')

if f1_results:
    retentions, accs = [], []
    for vname, vd in f1_results.items():
        try: num_parts = int(vname.split('-')[-1])
        except: continue
        if num_parts in retention_map:
            retentions.append(retention_map[num_parts])
            accs.append(vd['acc_mean'])
    
    if retentions:
        fig, ax = plt.subplots(figsize=(9,6))
        ax.scatter(retentions, accs, s=120, zorder=5, color='steelblue', edgecolors='black')
        for r, a, v in zip(retentions, accs, [f'{int(r*232965)} parts' for r in retentions]):
            ax.annotate(f'{int(1/r * 155)} parts' , (r,a), textcoords='offset points', xytext=(5,5), fontsize=9)
        # Regression line
        z = np.polyfit(retentions, accs, 1)
        p = np.poly1d(z)
        xs = np.linspace(min(retentions)-0.05, max(retentions)+0.05, 100)
        ax.plot(xs, p(xs), 'r--', alpha=0.7, label='Linear fit')
        ax.set_xlabel('Edge Retention Rate', fontsize=12)
        ax.set_ylabel('Test Accuracy', fontsize=12)
        ax.set_title('F1: ClusterGCN — Accuracy vs Edge Retention', fontsize=13)
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(SAVE_DIR,'F1_edge_retention.png'), dpi=150)
        plt.show()
else:
    print('F1 results not available yet.')